# 23. Loop Engineering

**Tier:** Agent Engineering
**Estimated time:** 65 minutes
**Prerequisites:** 19, 20, 22
**Priority:** 🔴 Crucial — the shift from prompting to designing systems; underlies 27b and P4. *If skipped, revisit when:* n/a.
**Source material:** @humzaakhalid — Loop Engineering (https://x.com/humzaakhalid/status/2064996712910041409) ; Karpathy framing via @suryanshti777 (https://x.com/suryanshti777/status/2057389330625339902)

## What You'll Learn
- The 3 failure modes of a single agent run: agentic laziness, self-preferential bias, goal drift
- All 6 loop patterns that fix them — with the **adversarial maker/checker** pattern in depth
- How an outer loop turns an unreliable single call into a dependable system

## Why This Matters
The shift from prompting to *loop engineering* is the shift from "ask the model and hope" to "design a system that prompts, checks, and re-prompts." A single agent run fails in predictable ways; a well-chosen loop catches those failures automatically. This is the capstone idea of the whole tier — and exactly what the P3 capstone is built on.


## Why a single run isn't enough: three failure modes

Karpathy's framing: you're no longer writing prompts, you're designing the *loop* that wraps the model. You need a loop because one model call fails in three reliable ways:

1. **Agentic laziness** — asked to do thorough work, the model does the minimum that looks done. "Review this code" gets you "looks good" instead of the bug on line 12.
2. **Self-preferential bias** — a model asked to grade its *own* output rates it generously. It can't be its own impartial judge, because the same blind spots that produced the error also evaluate it.
3. **Goal drift** — over a long run, the model gradually wanders from the original objective, optimizing for the last instruction instead of the actual goal.

The fix for all three is structural, not a better prompt: wrap the model in an outer loop that **generates, then independently checks, then re-generates**. The six patterns below are different shapes of that idea. The most important is **adversarial verification** — a *separate* call whose only job is to find what's wrong — because it directly defeats both laziness and self-preferential bias.


In [ ]:
import os, operator
from typing import TypedDict
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — pattern cells will be skipped.")

def ask(prompt, system="You are concise.", max_tokens=300, temperature=0.7):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    try:
        msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                      temperature=temperature, messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        return f"[skipped: {type(e).__name__}: {str(e)[:120]}]"


## Pattern 2 (first, because it matters most) — Adversarial Verification

A **maker** produces an answer; a *separate* **checker** call is told its job is to find flaws (not to be agreeable); the maker then revises. Two distinct calls with opposing incentives beat one call grading itself. We use a deliberately tricky task where a single pass often slips.


In [ ]:
TASK = ("Write a Python function `median(nums)` that returns the median of a list of numbers. "
        "Handle the even-length case correctly. Output only the function.")

def maker(task, feedback=None):
    prompt = task if feedback is None else f"{task}\n\nA reviewer found this issue, fix it:\n{feedback}"
    return ask(prompt, system="You are a careful Python developer.", max_tokens=300, temperature=0.3)

def checker(task, candidate):
    """A SEPARATE call whose only job is to find what's wrong — adversarial, not agreeable."""
    prompt = (f"Task: {task}\n\nCandidate solution:\n{candidate}\n\n"
              "You are a strict reviewer. Find the most important bug or edge case it gets wrong. "
              "If it is genuinely correct, reply exactly 'APPROVED'. Otherwise describe the single worst problem.")
    return ask(prompt, system="You are a ruthless code reviewer. Do not be agreeable.",
               max_tokens=200, temperature=0.0)

# Maker/checker loop: make -> check -> if not approved, remake with the critique.
candidate = maker(TASK)
print("=== initial draft ===\n", candidate[:300])
for round_num in range(1, 3):
    review = checker(TASK, candidate)
    print(f"\n=== checker round {round_num} ===\n", review[:250])
    if "APPROVED" in review.upper():
        print("\nChecker approved.")
        break
    candidate = maker(TASK, feedback=review)
    print(f"\n=== revised draft {round_num} ===\n", candidate[:300])


*The checker's separate, adversarial incentive is what surfaces the even-length bug a single self-graded pass tends to wave through — this is the antidote to both agentic laziness and self-preferential bias.*


## Pattern 1 — Fan-Out & Synthesize

Generate several independent takes in parallel, then merge. Diversity of attempts beats a single attempt (this is notebook 20's fan-out applied to *ideas*, and notebook 17's self-consistency generalized).


In [ ]:
def fan_out_synthesize(question, n=3):
    takes = [ask(f"Give one distinct angle on: {question}", temperature=1.0, max_tokens=80) for _ in range(n)]
    combined = "\n".join(f"- {t}" for t in takes)
    synthesis = ask(f"Merge these angles into one balanced recommendation:\n{combined}", max_tokens=150)
    return takes, synthesis

takes, synth = fan_out_synthesize("Should a 5-person startup adopt a multi-agent system?")
for i, t in enumerate(takes, 1):
    print(f"take {i}: {t[:90]}")
print("\nSYNTHESIS:", synth[:200])


## Pattern 3 — Tournament

Generate several candidates, judge them against a rubric, keep the winner. Unlike fan-out (which merges), a tournament *selects*.


In [ ]:
def tournament(task, n=3):
    candidates = [ask(task, temperature=1.0, max_tokens=100) for _ in range(n)]
    listing = "\n\n".join(f"Candidate {i+1}:\n{c}" for i, c in enumerate(candidates))
    verdict = ask(f"Task: {task}\n\n{listing}\n\nPick the single best candidate by number and say why in one line.",
                  system="You judge against clarity and correctness.", max_tokens=80, temperature=0.0)
    return candidates, verdict

cands, verdict = tournament("Write a one-line tagline for a robot-arm safety product.")
for i, c in enumerate(cands, 1):
    print(f"candidate {i}: {c[:80]}")
print("\nJUDGE:", verdict[:150])


## Pattern 4 — Loop Until Done (as a LangGraph cycle)

Iterate until a condition is satisfied or a budget runs out. This is where LangGraph's **conditional edge that loops back** shines — a true cycle, not a straight line. The state carries an iteration counter so the loop can't run forever.


In [ ]:
from langgraph.graph import StateGraph, START, END

class LoopState(TypedDict):
    target_words: int
    draft: str
    iterations: int

def refine(state):
    n = len(state["draft"].split())
    instruction = (f"Rewrite to be EXACTLY about {state['target_words']} words "
                   f"(current: {n}). Keep the meaning. Topic: industrial robots.")
    new_draft = ask(instruction if state["draft"] else
                    f"Write ~{state['target_words']} words about industrial robots.", max_tokens=200)
    return {"draft": new_draft, "iterations": state["iterations"] + 1}

def is_done(state):
    n = len(state["draft"].split())
    # Done if within 20% of target, or we've spent the budget.
    if abs(n - state["target_words"]) <= 0.2 * state["target_words"] or state["iterations"] >= 3:
        return END
    return "refine"

loop = StateGraph(LoopState)
loop.add_node("refine", refine)
loop.add_edge(START, "refine")
loop.add_conditional_edges("refine", is_done, {"refine": "refine", END: END})
loop_graph = loop.compile()

if HAS_ANTHROPIC:
    out = loop_graph.invoke({"target_words": 40, "draft": "", "iterations": 0})
    print(f"Finished after {out['iterations']} iteration(s); final length = {len(out['draft'].split())} words")
    print(out["draft"][:250])
else:
    print("  [skipped: no ANTHROPIC_API_KEY] — graph built:", [n for n in loop_graph.get_graph().nodes if not n.startswith('__')])


## Patterns 5 & 6 — Generate & Filter, and Deep Verification

**Generate & Filter**: over-generate, then keep only items passing a predicate. **Deep Verification**: don't just check the answer — check it along several independent dimensions, so a flaw on any axis is caught.


In [ ]:
# Pattern 5 — Generate & Filter
def generate_and_filter(prompt, n=4, predicate=lambda s: len(s.split()) <= 8):
    items = [ask(prompt, temperature=1.0, max_tokens=40) for _ in range(n)]
    kept = [s for s in items if predicate(s.strip())]
    return items, kept

generated, kept = generate_and_filter("Suggest a short name for a warehouse robot. Name only.")
print("generated:", [g.strip()[:30] for g in generated])
print("kept (<= 8 words):", [k.strip()[:30] for k in kept])

# Pattern 6 — Deep Verification (check several axes independently)
def deep_verify(claim):
    axes = {
        "factual": f"Is this factually plausible? One word yes/no + 3-word reason: {claim}",
        "internally_consistent": f"Is this internally consistent? One word yes/no + 3-word reason: {claim}",
        "complete": f"Does this fully answer, or is something missing? 5 words max: {claim}",
    }
    return {axis: ask(q, max_tokens=30, temperature=0.0) for axis, q in axes.items()}

checks = deep_verify("The Helios X1 has a 2-year warranty and a 10-hour battery, ideal for 24/7 operation.")
for axis, verdict in checks.items():
    print(f"{axis:22s}: {verdict[:70]}")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

patterns = ["Fan-Out &\nSynthesize", "Adversarial\nVerification", "Tournament",
            "Loop Until\nDone", "Generate\n& Filter", "Deep\nVerification"]
fixes = ["laziness", "laziness + bias", "bias", "goal drift", "laziness", "laziness + bias"]
plt.figure(figsize=(9, 4))
plt.bar(patterns, [1]*6, color=["#4C72B0", "#C44E52", "#DD8452", "#55A868", "#8172B3", "#937860"])
for i, f in enumerate(fixes):
    plt.text(i, 0.5, f, ha="center", va="center", color="white", fontsize=8, rotation=90)
plt.title("The 6 loop patterns and the failure mode each primarily fixes")
plt.yticks([]); plt.tight_layout(); plt.show()


*Each pattern is a different shape of the same idea — generate, check independently, regenerate — and adversarial verification is the one that attacks two failure modes at once.*


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Make the checker stricter
# Task: Change the checker's system prompt to also require the function handle an EMPTY list. Re-run
#       the adversarial loop. Does it now flag the empty-list case the first checker ignored?
# Hint: The checker only catches what you point it at — its instructions define how adversarial it is.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Tournament with an explicit rubric
# Task: Modify `tournament` to score each candidate 1-5 on TWO named criteria, then pick the highest
#       total. Does a rubric change which candidate wins vs. the vague 'best' judge?
# Hint: Ask the judge to return the scores, then parse them — this is more reliable than 'pick the
#       best', and it's how an LLM-as-judge eval (Tier 5, notebook 26) works.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Compose two patterns
# Task: Build a mini-pipeline that runs a Tournament to pick the best draft, then feeds the winner
#       through one round of Adversarial Verification. Which combination would you trust for a
#       high-stakes output?
# Hint: This generate-then-verify composition is the backbone of the P3 capstone's report step —
#       select the best, then have a separate checker sign off on it.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
def strict_checker(task, candidate):
    prompt = (f"Task: {task}\nCandidate:\n{candidate}\n"
              "Strict review. It MUST handle even-length lists AND empty lists. "
              "Reply 'APPROVED' only if both hold, else name the worst gap.")
    return ask(prompt, system="You are a ruthless reviewer.", max_tokens=150, temperature=0.0)
print(strict_checker(TASK, candidate))   # now flags the empty-list path

# Exercise 2
def tournament_rubric(task, n=3):
    cands = [ask(task, temperature=1.0, max_tokens=100) for _ in range(n)]
    listing = "\n\n".join(f"Candidate {i+1}:\n{c}" for i, c in enumerate(cands))
    verdict = ask(f"{task}\n\n{listing}\n\nScore each candidate 1-5 on CLARITY and CORRECTNESS, "
                  "then name the highest total.", max_tokens=150, temperature=0.0)
    return cands, verdict
print(tournament_rubric("Write a tagline for a robot-arm safety product.")[1][:200])

# Exercise 3
def tournament_then_verify(task):
    cands, verdict = tournament(task)
    # naive: take candidate 1; in practice parse the winner index from `verdict`
    winner = cands[0]
    review = checker(task, winner)
    final = winner if "APPROVED" in review.upper() else maker(task, feedback=review)
    return final
# Tournament finds the best of several; adversarial verification then guards against a shared flaw.
# For high-stakes output, trust generate-many + independent-verify over any single pass.
```
</details>


## Key Takeaways
- A single model call fails predictably: agentic laziness, self-preferential bias, goal drift.
- The structural fix is an outer loop: generate → independently check → regenerate.
- Adversarial verification (a *separate* checker with opposing incentives) is the most important pattern — it defeats laziness and self-preferential bias at once.
- The six patterns are shapes of one idea: Fan-Out & Synthesize, Adversarial Verification, Tournament, Loop Until Done, Generate & Filter, Deep Verification.
- LangGraph's conditional edge makes "loop until done" a real cycle with a built-in budget; the patterns compose (tournament → verify) for high-stakes work.

## What's Next
That completes Tier 4. The **P3 capstone** (in `06_projects/`) puts it all together: a research agent built from the raw loop (19), orchestrated with LangGraph (20), traced with LangSmith (21), with a managed context (22) and an adversarial verification loop (23) guarding its final report.
